# Task 2.2 Reproduction of any one Contribution from the Selected Paper

**Objective:** The result I am attempting to reproduce is the core "SCW-I" (Soft Confidence-Weighted with linear penalty) algorithm as defined in **Proposition 1** of the paper. I will demonstrate its capability to learn online and predict effectively on a noisy binary classification dataset.

**Evaluation Metric:** The evaluation metric will be the **Online Cumulative Mistake Rate** (the total number of prediction mistakes made divided by the total number of instances seen so far) and the final accuracy on a holdout test set.

In [1]:
import numpy as np
from scipy.stats import norm

This code block imports necessary libraries. We need `norm` from `scipy.stats` to calculate $\phi = \Phi^{-1}(\eta)$.

*Cites:* Section 3, definitions related to Confidence-Weighted learning.

In [2]:
class SCW1:
    def __init__(self, n_features, C=1.0, eta=0.9):
        self.C = C
        self.eta = eta
        self.phi = norm.ppf(eta)
        self.psi = 1 + (self.phi ** 2) / 2
        self.zeta = 1 + self.phi ** 2
        
        self.mu = np.zeros(n_features)
        self.Sigma = np.eye(n_features)

This constructor initializes the `SCW1` class, setting the initial mean vector $\mu_0 = (0, \dots, 0)^T$ and the covariance matrix $\Sigma_0 = I$. It also pre-computes constants $\psi$, $\zeta$, and $\phi$ based on the confidence threshold $\eta$.

*Cites:* Algorithm 1 "INITIALIZATION" steps, and Proposition 1 definitions.

In [3]:
    def predict(self, x):
        return np.sign(np.dot(self.mu, x))

This method predicts the class label $\hat{y}_t$ for an input $x_t$ by taking the sign of the dot product between the mean weight vector $\mu_{t-1}$ and the input.

*Cites:* Algorithm 1, "Make prediction: $\hat{y}_t = \text{sgn}(\mu_{t-1} \cdot x_t)$".

In [4]:
    def update(self, x, y):
        m_t = y * np.dot(self.mu, x)
        v_t = np.dot(x, np.dot(self.Sigma, x))
        
        loss = max(0, self.phi * np.sqrt(v_t) - m_t)

        if loss > 0:
            alpha_t = min(self.C, max(0, (1 / (v_t * self.zeta)) * (-m_t * self.psi + np.sqrt( (m_t**2 * self.phi**4)/4 + v_t * self.phi**2 * self.zeta ))))
            
            u_t = 0.25 * (-alpha_t * v_t * self.phi + np.sqrt( (alpha_t * v_t * self.phi)**2 + 4 * v_t ))**2
            if u_t < 0: return
            beta_t = (alpha_t * self.phi) / (np.sqrt(u_t) + v_t * alpha_t * self.phi) if (np.sqrt(u_t) + v_t * alpha_t * self.phi) != 0 else 0
            
            Sigma_x = np.dot(self.Sigma, x)
            self.mu = self.mu + alpha_t * y * Sigma_x
            self.Sigma = self.Sigma - beta_t * np.outer(Sigma_x, Sigma_x)

This calculates the margin $m_t$ and variance $v_t$ for the instance, then evaluates the loss $\ell_\phi$. If the loss is positive, it executes the strict closed-form updates for $\alpha_t$ and $\beta_t$, and updates the model's mean $\mu$ and covariance $\Sigma$ matrices.

*Cites:* Section 3, loss definition, and **Proposition 1** closed-form solution formulas.

### Running the Online Training Loop
Below we combine the dummy dataset generated in Task 2.1 to simulate the online streaming environment.

In [5]:
if 'X_train' not in locals():
    # Fallback initialization if run out-of-order
    from sklearn.datasets import make_classification
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler
    np.random.seed(42)
    X, y = make_classification(n_samples=1000, n_features=20, n_informative=15, 
                               n_redundant=2, n_classes=2, flip_y=0.1, random_state=42)
    y = np.where(y == 0, -1, 1)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

class SCW1Full:
    def __init__(self, n_features, C=1.0, eta=0.9):
        self.C = C
        self.eta = eta
        self.phi = norm.ppf(eta)
        self.psi = 1 + (self.phi ** 2) / 2
        self.zeta = 1 + self.phi ** 2
        self.mu = np.zeros(n_features)
        self.Sigma = np.eye(n_features)
        
    def predict(self, x):
        return np.sign(np.dot(self.mu, x))
        
    def update(self, x, y):
        m_t = y * np.dot(self.mu, x)
        v_t = np.dot(x, np.dot(self.Sigma, x))
        loss = max(0, self.phi * np.sqrt(v_t) - m_t)
        if loss > 0:
            alpha_t = min(self.C, max(0, (1 / (v_t * self.zeta)) * (-m_t * self.psi + np.sqrt( (m_t**2 * self.phi**4)/4 + v_t * self.phi**2 * self.zeta ))))
            u_t = 0.25 * (-alpha_t * v_t * self.phi + np.sqrt( (alpha_t * v_t * self.phi)**2 + 4 * v_t ))**2
            if u_t < 0: return
            beta_t = (alpha_t * self.phi) / (np.sqrt(u_t) + v_t * alpha_t * self.phi) if (np.sqrt(u_t) + v_t * alpha_t * self.phi) != 0 else 0
            Sigma_x = np.dot(self.Sigma, x)
            self.mu = self.mu + alpha_t * y * Sigma_x
            self.Sigma = self.Sigma - beta_t * np.outer(Sigma_x, Sigma_x)

model = SCW1Full(n_features=X_train.shape[1], C=1.0, eta=0.9)
mistakes = 0
mistake_rates = []

for i in range(len(X_train)):
    x_t = X_train[i]
    y_t = y_train[i]
    
    # Predict
    y_pred = model.predict(x_t)
    if y_pred == 0: # handle explicit 0 from sign
        y_pred = 1
        
    # Evaluate mistake
    if y_pred != y_t:
        mistakes += 1
        
    # Train
    model.update(x_t, y_t)
    
    mistake_rates.append(mistakes / (i + 1))

# Evaluate on hold-out test set
test_mistakes = 0
for i in range(len(X_test)):
    if np.sign(np.dot(model.mu, X_test[i])) != y_test[i]:
        test_mistakes += 1

print(f"Final Online Mistake Rate: {mistake_rates[-1]:.4f}")
print(f"Hold-out Test Accuracy: {1 - (test_mistakes / len(X_test)):.4f}")


Final Online Mistake Rate: 0.2600
Hold-out Test Accuracy: 0.7650


This runs the online learning loop, simulating the arrival of instances one by one, measuring cumulative errors, and updating the model. It then checks final accuracy on a test set.

*Cites:* Section 2.1 (Overview of Online Learning) loop execution.